In [1]:
import pandas as pd
import duckdb
import os
import glob

In [2]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

In [ ]:
# ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
# ...OR VICE VERSA?

# THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
# THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
# OR THEY ARE A COMINBATION OF MULTIPLE PARENT PARTS.
# EITHER WAY, THE PARENT(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

# TABLES NEEDED (6 out of 12):

    # part_relationships - core table to identify parent-child connections
    # parts - to get the part name, for human readability
    # part_categories - allow for more granular analysis by part category
    # inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


#---note: the 'inventory_sets' table is not needed as it just records the quantity of a given part used per set, which is not needed here

In [3]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
sets = dfs["sets"]

In [9]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [22]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
joined = duckdb.sql("""
                    SELECT
                        pc.name AS category,                         --only needs to be shows once, as parent and child likely to have same category 
                        pr.parent_part_num,
                        p_p.name AS parent_part_name,
                        c_p.name AS child_part_name,
                        pr.child_part_num
                    FROM part_relationships pr
                    INNER JOIN parts c_p ON pr.child_part_num = c_p.part_num
                    INNER JOIN parts p_p ON pr.parent_part_num = p_p.part_num
                    INNER JOIN part_categories pc ON c_p.part_cat_id = pc.id
                    ORDER BY pc.id
                    """).df()

In [24]:
joined.head(20)

,category,parent_part_num,parent_part_name,child_part_name,child_part_num
0,Baseplates,2746,"Storage Case Container, Build-N-Store Chest, 3...",Baseplate 16 x 32,3857
1,Bricks Sloped,75538,"Slope Curved 8 x 8 x 4, Ramp",Slope 8 x 5 x 2/3 with 2 x 6 Studs,75539
2,Bricks Sloped,30363pr0010,Slope 18° 4 x 2 with Ferrari Headlight Left Print,Slope 18° 4 x 2 with Ferrari Headlight Right P...,30363pr0011
3,"Duplo, Quatro and Primo",61649,"Duplo Door / Window Frame Flat Front Surface, ...",Duplo Door / Window with Porthole and 'BOAT YA...,4248pr0001
4,"Duplo, Quatro and Primo",4883c01,"Duplo Car Base 2 x 6 with Yellow Wheels, (Old ...",Duplo Car Body (Old Style),dupupn0017
5,"Duplo, Quatro and Primo",31063,Duplo Boat Hull 6 x 16 Top Section with 4 x 4 ...,Duplo Boat Hull 6 x 16 Bottom Section,71402
6,"Duplo, Quatro and Primo",11345,Duplo Building Door Frame 4 x 4 x 3,Duplo Door / Window with Porthole and 'BOAT YA...,4248pr0001
7,"Duplo, Quatro and Primo",4253,Duplo Door / Window Frame Flat Front Surface,Duplo Door / Window with Porthole and 'BOAT YA...,4248pr0001
8,"Duplo, Quatro and Primo",2204pr0001,Duplo Building 6 x 8 x 6 with Front Door and W...,Duplo Door / Window with Porthole and 'BOAT YA...,4248pr0001
9,"Duplo, Quatro and Primo",2332,Duplo Door / Window Frame with Raised Door Out...,Duplo Door / Window with Porthole and 'BOAT YA...,4248pr0001
